# Inteligencia Artificial 2026 - Lab 03

En este laboratorio vamos a diseñar redes neuronales *fully connected* en la librería `tensorflow.keras` para un problema de clasificación y un problema de regresión.

### Ejercicio 1: Clasificación de Dígitos (Scikit-Learn digits)
Este ejercicio se centra en entender cómo una red neuronal "ve" patrones espaciales aplanados.

**Dataset:** `sklearn.datasets.load_digits()` contiene 1,797 muestras, 64 atributos.
**Objetivo:** Configurar la capa de salida y la función de pérdida para clasificación multiclase (10 dígitos).

In [ ]:
!pip install tensorflow

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.datasets import load_digits, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score

# TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Configuración de estilo para gráficas
sns.set_theme(style="whitegrid")

### Ejercicio 1: Clasificación de Dígitos (Scikit-Learn digits)
Este ejercicio se centra en entender cómo una red neuronal "ve" patrones espaciales aplanados.

**Dataset:** `sklearn.datasets.load_digits()` contiene 1,797 muestras, 64 atributos.

**Objetivo:** Configurar la capa de salida y la función de pérdida para clasificación multiclase (10 dígitos).

**Instrucciones:**
* Normalizar los datos dividiendo por 16 (el valor máximo de los píxeles). Debe explicar las transformaciones que se están haciendo tanto al conjunto de datos, como a las etiquetas.

In [ ]:
# 1. Carga de datos
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# 2. Partición de datos (Train/Test)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42, stratify=y_digits
)

# 3. Normalización de los datos de entrada
X_train_d_norm = X_train_d / 16.0
X_test_d_norm = X_test_d / 16.0

# 4. Transformación de las etiquetas (One-Hot Encoding)
y_train_d_cat = to_categorical(y_train_d, num_classes=10)
y_test_d_cat = to_categorical(y_test_d, num_classes=10)

**Explicación del Procedimiento y Transformaciones:**
* **Partición (Train/Test):** Se utilizó una proporción de 80% para entrenamiento y 20% para pruebas (`test_size=0.2`). Esta distribución asegura suficientes datos para que el modelo aprenda patrones, manteniendo un conjunto representativo para evaluar su generalización. Además, se incluyó `stratify=y_digits` para garantizar que ambos conjuntos tengan la misma proporción de cada clase, evitando sesgos.
* **Conjunto de datos (X):** Las imágenes originales tienen valores de píxeles que van de 0 a 16. Al dividir todo entre 16, normalizamos los datos a un rango de [0, 1]. Esto estabiliza los gradientes durante el entrenamiento y ayuda a que la red neuronal converja eficientemente.
* **Etiquetas (y):** Las etiquetas originales son enteros del 0 al 9. Se aplicó una transformación *One-Hot Encoding* (usando `to_categorical`), convirtiendo cada entero en un vector de tamaño 10 con un '1' en la posición de la clase y '0' en el resto. Es obligatorio debido a que la red utiliza la función *Softmax* para predecir probabilidades multiclase independientes.



* Diseñar una arquitectura con al menos 2 capas ocultas. En cada capa, utilizar las funciones de activación adecuadas. Justifique la elección del número de capas y de neuronas, así como de las funciones de activación.

In [ ]:
model_digits = Sequential([
    Dense(64, activation='relu', input_shape=(64,)),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

model_digits.summary()

**Justificación de la Arquitectura:**
* **Capas y Neuronas:** Se eligieron 2 capas ocultas. La primera con 64 neuronas (coincidiendo con los 64 atributos de entrada) para capturar las características iniciales, y una segunda de 32 neuronas para una compresión progresiva. La capa de salida tiene exactamente 10 neuronas, una por cada clase a predecir.
* **Funciones de Activación:** En las capas ocultas se utilizó **ReLU**, ya que mitiga el problema del desvanecimiento del gradiente y permite un entrenamiento rápido. En la capa de salida se utilizó **Softmax**, la función matemática estándar para clasificación multiclase, que transforma las salidas numéricas brutas en probabilidades que suman 1.

* Elegir la función de pérdida apropiada, así como los hiper-parámetros que considere más adecuados. Justifique la elección de estos hiper-parámetros.

In [ ]:
# Compilación del modelo
model_digits.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenamiento del modelo
history_digits = model_digits.fit(
    X_train_d_norm, y_train_d_cat,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

**Justificación de Hiper-parámetros:**
* **Función de pérdida:** Se utilizó `categorical_crossentropy`, que es la estándar para clasificación multiclase con etiquetas One-Hot. Penaliza fuertemente al modelo si asigna alta probabilidad a una clase incorrecta.
* **Optimizador y Learning Rate:** Se eligió **Adam** con $\alpha = 0.001$. Adam ajusta automáticamente la tasa de aprendizaje por cada parámetro. El valor de 0.001 es el default recomendado ya que suele ofrecer una excelente y rápida convergencia sin saltarse el mínimo global.
* **Epochs, Batch Size y Validation:** Se entrenó por 50 *epochs*, suficientes para este pequeño volumen de datos. El `batch_size=32` procesa los datos en bloques pequeños, ofreciendo un buen equilibrio entre uso de memoria y regularización del gradiente estocástico. El `validation_split=0.1` reserva un 10% del set de entrenamiento para monitorear el *overfitting* en cada iteración.


* Mostrar una Matriz de Confusión usando seaborn para identificar qué números confunde la red (comúnmente debería ser el 1 con el 7 o el 3 con el 8).


In [ ]:
# Predicciones
y_pred_d_probs = model_digits.predict(X_test_d_norm)
y_pred_d_classes = np.argmax(y_pred_d_probs, axis=1)

# Matriz de Confusión
cm = confusion_matrix(y_test_d, y_pred_d_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=digits.target_names,
            yticklabels=digits.target_names)
plt.title('Matriz de Confusión - Clasificación de Dígitos')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

* Muestre un conjunto de métricas de desempeño (tanto en entrenamiento como en test), para determinar la efectividad de su clasificador. Muestre 5 ejemplos de imágenes bien clasificadas y 5 ejemplos de imágenes mal clasificadas.

In [ ]:
# Métricas de Desempeño
train_loss, train_acc = model_digits.evaluate(X_train_d_norm, y_train_d_cat, verbose=0)
test_loss, test_acc = model_digits.evaluate(X_test_d_norm, y_test_d_cat, verbose=0)

print(f"Métricas en Entrenamiento -> Pérdida: {train_loss:.4f}, Exactitud (Accuracy): {train_acc:.4f}")
print(f"Métricas en Prueba (Test) -> Pérdida: {test_loss:.4f}, Exactitud (Accuracy): {test_acc:.4f}\n")
print("Reporte de Clasificación (Test):\n", classification_report(y_test_d, y_pred_d_classes))

# Índices para ejemplos visuales
correct_indices = np.where(y_pred_d_classes == y_test_d)[0]
incorrect_indices = np.where(y_pred_d_classes != y_test_d)[0]

# Función para graficar ejemplos
def plot_examples(indices, title):
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    fig.suptitle(title, fontsize=16)
    for i, ax in enumerate(axes):
        if i < len(indices):
            idx = indices[i]
            image = X_test_d[idx].reshape(8, 8)
            ax.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
            ax.set_title(f"Real: {y_test_d[idx]}\nPred: {y_pred_d_classes[idx]}")
        ax.axis('off')
    plt.show()

# Mostrar 5 bien clasificadas y 5 mal clasificadas
plot_examples(correct_indices[:5], "5 Ejemplos Bien Clasificados")

if len(incorrect_indices) > 0:
    plot_examples(incorrect_indices[:5], "Ejemplos Mal Clasificados")
else:
    print("¡Impresionante, el modelo no se ha equivocado en ninguna predicción!")

## Ejercicio 2: Regresión de Precios (Scikit-Learn california_housing)
Este ejercicio se centra en construir un modelo de regresión mediante una red neuronal densa.

**Dataset:** `sklearn.datasets.fetch_california_housing()`

**Objetivo:** Manejo de escalas heterogéneas y métricas de error continuo.

**Instrucciones:**
* Estandarizar los datos (Escalamiento Obligatorio). Usar StandardScaler en los datos de entrada. Sin esto, la red tardará demasiado en converger o los gradientes explotarán.

In [ ]:
# Cargar daros
housing = fetch_california_housing()
X_house = housing.data
y_house = housing.target


X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)
# Estandarización
scaler = StandardScaler()
X_train_h_scaled = scaler.fit_transform(X_train_h)
X_test_h_scaled = scaler.transform(X_test_h)

**Explicación de la Estandarización:**
 Al usar `StandardScaler`, transformamos las distribuciones para que tengan una media de 0 y desviación estándar de 1.
 Esto es vital en redes neuronales porque si un atributo tiene valores en los miles y otro en decimales, los pesos de la red oscilarán bruscamente, causando que los gradientes exploten o que el algoritmo de optimización jamás logre converger al mínimo de la función de pérdida.

---

* Probar una red más profunda (ej. 3 capas: 64, 32, 16 o similar) dada la complejidad de los datos socioeconómicos. Justificar la elección de su arquitectura.

In [ ]:
model_housing = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_h_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

model_housing.summary()

**Justificación de Arquitectura:**
* **Capas:** Se utilizan 3 capas ocultas (64, 32, 16). Los datos socioeconómicos presentan relaciones fuertemente no lineales y complejas para determinar un precio. Una red más profunda permite extraer jerarquías de características latentes más abstractas.
* **Activación:** Se mantiene ReLU para las capas ocultas por eficiencia. La capa de salida tiene 1 sola neurona con activación Lineal (o sin activación explícita), ya que en un problema de regresión necesitamos predecir un valor continuo que puede tomar cualquier rango (en este caso, el precio de la casa en cientos de miles de dólares).
---

* Elegir la función de pérdida apropiada (ej. MSE o MAE) y usar un optimizador Adam. Justifique la elección de sus hiper-parámetros.

In [ ]:
model_housing.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='mse',
    metrics=['mae']
)

history_housing = model_housing.fit(
    X_train_h_scaled, y_train_h,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=0
)

**Justificación de Hiper-parámetros:**
* **Función de pérdida:** Se escogió **MSE (Error Cuadrático Medio)** porque castiga de manera cuadrática los errores grandes, obligando al modelo a no equivocarse drásticamente en los precios de las casas.
* **Optimizador:** **Adam** con $\alpha = 0.005$. Dado que hay más capas y datos, un learning rate ligeramente modificado ayuda a sortear mínimos locales.
* **Batch Size y Epochs:** Al ser un dataset con más de 20,000 registros, un batch size de 64 procesa la información en bloques eficientes. Se entrenó por 100 *epochs* asumiendo que un dataset más complejo requiere más iteraciones para el descenso del gradiente.

---

* Mostrar un Generar un Scatter-Plot de Predicciones vs. Valores Reales.

Muestre un conjunto de métricas de desempeño (tanto en entrenamiento como en test), para determinar la efectividad de su regresión. Elabore 3 ejemplos de predicción con su modelo de regresión para 3 observaciones nuevas (que no son parte del dataset).

In [ ]:
# Predicciones
y_pred_h_train = model_housing.predict(X_train_h_scaled)
y_pred_h_test = model_housing.predict(X_test_h_scaled)

# Métricas
print("Métricas en Entrenamiento:")
print(f"MSE: {mean_squared_error(y_train_h, y_pred_h_train):.4f}")
print(f"MAE: {mean_absolute_error(y_train_h, y_pred_h_train):.4f}")
print(f"R2 Score: {r2_score(y_train_h, y_pred_h_train):.4f}\n")

print("Métricas en Prueba (Test):")
print(f"MSE: {mean_squared_error(y_test_h, y_pred_h_test):.4f}")
print(f"MAE: {mean_absolute_error(y_test_h, y_pred_h_test):.4f}")
print(f"R2 Score: {r2_score(y_test_h, y_pred_h_test):.4f}\n")

# Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test_h, y_pred_h_test, alpha=0.3, color='orange')
plt.plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()], 'k--', lw=2)
plt.title('Predicciones vs Valores Reales (California Housing)')
plt.xlabel('Precio Real')
plt.ylabel('Precio Predicho')
plt.show()

nuevas_casas = np.array([
    [5.0, 25.0, 6.0, 1.0, 1000.0, 3.0, 37.5, -122.0], # Casa en zona central, estándar
    [1.5, 40.0, 3.0, 1.2, 500.0, 4.0, 36.0, -119.0],  # Ingreso bajo, casa antigua
    [8.0, 10.0, 8.0, 1.5, 2000.0, 2.5, 34.0, -118.0]  # Ingreso alto, casa nueva, muchas hab
])

nuevas_casas_scaled = scaler.transform(nuevas_casas)
predicciones_nuevas = model_housing.predict(nuevas_casas_scaled)

print("Predicciones para 3 observaciones nuevas:")
for i, pred in enumerate(predicciones_nuevas):
    print(f"Casa nueva {i+1} -> Precio estimado: ${pred[0]*100000:,.2f}")